# Session 1: why a flat map of a round Earth must distort
**A. Real-world problem.** Peel an orange in one piece and try to press the peel flat on a
table. It tears, or it must be stretched. The Earth is a ball, and every flat map is a pressed
peel. In this session you measure how much the simplest map (the equirectangular map, latitude
and longitude drawn as a plain grid) stretches the world at different latitudes.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
R_EARTH = 6371.0   # mean radius of the Earth in km (a sphere is our simplified model)

## Step 1: latitude, longitude and the equirectangular map
*Latitude* is the angle north (+) or south (-) of the equator, from -90 to +90 degrees.
*Longitude* is the angle east (+) or west (-) of a reference meridian, from -180 to +180.
The equirectangular map simply uses them as grid coordinates:

    x = R * longitude,      y = R * latitude        (angles in radians, x and y in km)

In [ ]:
def equirect(lat_deg, lon_deg, R=R_EARTH):
    """Map position (x, y) in km of a point given by latitude and longitude in degrees."""
    return R * np.radians(lon_deg), R * np.radians(lat_deg)

## Step 2: how long is a circle of latitude?
On the ball, the circle at latitude `lat` is smaller than the equator: its radius is
`R cos(lat)`. On the map every such circle is drawn with the same length as the equator.

In [ ]:
def parallel_length_sphere(lat_deg, R=R_EARTH):
    """True length in km of the full circle of latitude."""
    return 2 * np.pi * R * np.cos(np.radians(lat_deg))


def parallel_length_map(lat_deg, R=R_EARTH):
    """Length in km that the same circle has on the equirectangular map (the map width)."""
    return 2 * np.pi * R


def ew_stretch(lat_deg):
    """East-west stretch of the equirectangular map: map length / true length = 1 / cos(lat)."""
    lat = abs(float(lat_deg))
    if lat >= 90:
        return float("inf")      # a pole is a single point, drawn as a whole line
    return 1.0 / np.cos(np.radians(lat))


def gap_fraction(lat_deg):
    """Share of a map row that is 'made-up' (stretched) material: 1 - cos(lat)."""
    return 1.0 - np.cos(np.radians(lat_deg))

## Step 3: a patch of the world
A patch that is `span` degrees wide in longitude has true east-west width
`R cos(lat) * span` but is drawn `R * span` wide. North-south it is drawn at true size, so
**area on the map = area on the ball x 1 / cos(lat)**.

In [ ]:
def patch_width_true(lat_deg, span_deg, R=R_EARTH):
    return R * np.cos(np.radians(lat_deg)) * np.radians(span_deg)


def patch_width_map(span_deg, R=R_EARTH):
    return R * np.radians(span_deg)


def area_stretch(lat_deg):
    """Area on the map divided by true area for the equirectangular map."""
    return ew_stretch(lat_deg)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    # Figure 1: the orange peel.  12 gores of 30 degrees; each is only cos(lat) wide.
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 6.2))
    lat = np.linspace(-90, 90, 181)
    for k in range(12):
        lo = -180 + 30 * k
        c = lo + 15
        half = 15 * np.cos(np.radians(lat))
        a1.fill_betweenx(lat, c - half, c + half, color="#e69f00", alpha=0.75, lw=0.5, ec="white")
        a2.fill_betweenx(lat, lo, lo + 30, color="#e69f00", alpha=0.75, lw=0.5, ec="white")
    a1.set_title("Peel the ball into 12 strips: each strip is only cos(latitude) wide (gaps appear)")
    a2.set_title("Press them into a plain grid: the gaps must be filled by stretching")
    for ax in (a1, a2):
        ax.set_ylabel("latitude (degrees)")
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
        ax.set_aspect("equal")
    a2.set_xlabel("longitude (degrees)")
    fig.tight_layout()
    fig.savefig(OUT / "orange_peel.png", dpi=120)

    # Figure 2: stretch against latitude
    lats = np.linspace(0, 80, 161)
    fig2, ax = plt.subplots(figsize=(6, 4))
    ax.plot(lats, [ew_stretch(v) for v in lats], color="#0072b2", label="east-west stretch = area stretch")
    ax.axhline(1, color="gray", ls=":", label="true size")
    ax.plot([60], [ew_stretch(60)], "ro")
    ax.annotate("60 degrees: x2", (60, 2), textcoords="offset points", xytext=(-80, 12))
    ax.set_xlabel("latitude (degrees)")
    ax.set_ylabel("map size / true size (no unit)")
    ax.set_title("Equirectangular map: stretch grows toward the poles")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "stretch_vs_latitude.png", dpi=120)
    for v in (0, 30, 60, 80):
        print(f"latitude {v:2d}: circle of latitude {parallel_length_sphere(v):9.1f} km on the ball, "
              f"{parallel_length_map(v):9.1f} km on the map, stretch {ew_stretch(v):.3f}")

## Where this stops
Stretching only east-west is one choice. Other maps stretch differently (Session 2 gives
Mercator) and Session 3 measures every kind of distortion with one tool, the Jacobian.

## Coding tasks
1. **Easy.** Change the latitude in `lesson.py` to 45 and 75 degrees. Predict the stretch with your calculator before you run it.
2. **Stretch.** Write `latitude_for_stretch(s)` that returns the latitude (in degrees) where the east-west stretch equals s. Check that it returns 60 for s = 2.
3. **Explain.** Explain, using the orange, why the stretch becomes infinite at the pole. Why is it still sensible to say the pole is a single point?

## Assessment
- **A. Conceptual.** Why can the peel of an orange not be pressed flat without tearing or stretching? What does this mean for every flat map of the Earth?
- **B. Numerical.** At 45 degrees north, what is the true length of 10 degrees of longitude, how long does the equirectangular map draw it, and what is the stretch? Use R = 6371 km.
- **C. Graph and simulation interpretation.** In the stretch graph, read the stretch at 60 degrees and at 80 degrees. By how many times does the area of a country of fixed size grow between 60 and 80 degrees on this map?
- **D. Design and optimisation challenge.** A weather office wants a flat grid map of a region in which east-west lengths are never stretched by more than 25 percent. What is the highest latitude the region may reach? What would you do for a region that reaches 70 degrees?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    x, y = L.equirect(0.0, 0.0)
    checks.append(("the point (0, 0) maps to the map origin", abs(x) < 1e-9 and abs(y) < 1e-9))
    x, y = L.equirect(90.0, 180.0)
    checks.append(("lat 90 lon 180 maps to (pi R, pi R / 2)",
                   abs(x - np.pi * L.R_EARTH) < 1e-6 and abs(y - np.pi * L.R_EARTH / 2) < 1e-6))
    checks.append(("the equator is drawn at true length (stretch 1)", abs(L.ew_stretch(0.0) - 1.0) < 1e-12))

    # worked example: the circle of latitude at 60 degrees north
    true60 = L.parallel_length_sphere(60.0)
    map60 = L.parallel_length_map(60.0)
    checks.append((f"worked example: 60 degrees N circle is {true60:.1f} km, the map draws {map60:.1f} km, stretch 2",
                   abs(true60 - 20015.1) < 0.2 and abs(map60 - 40030.2) < 0.2 and abs(map60 / true60 - 2.0) < 1e-9))

    # hand calculation: 1 degree of longitude at 0 and at 60 degrees
    one0 = L.patch_width_true(0.0, 1.0)
    one60 = L.patch_width_true(60.0, 1.0)
    checks.append(("1 degree of longitude is 111.19 km at the equator", abs(one0 - 111.19) < 0.01))
    checks.append(("1 degree of longitude is half of that at 60 degrees", abs(one60 - one0 / 2) < 1e-9))
    checks.append(("the map draws every degree of longitude the same length", abs(L.patch_width_map(1.0) - one0) < 1e-9))

    # stretch grows toward the poles; symmetric in north and south
    lats = np.arange(0, 85, 5.0)
    s = np.array([L.ew_stretch(v) for v in lats])
    checks.append(("stretch never decreases toward the pole", bool(np.all(np.diff(s) > 0))))
    checks.append(("north and south are symmetric", abs(L.ew_stretch(-45.0) - L.ew_stretch(45.0)) < 1e-12))
    checks.append(("area stretch equals east-west stretch", abs(L.area_stretch(50.0) - L.ew_stretch(50.0)) < 1e-12))

    # edge cases: a pole and bad rounding
    checks.append(("at the pole the stretch is infinite (a point drawn as a line)", L.ew_stretch(90.0) == float("inf")))
    checks.append(("gap fraction is 0 at the equator and 0.5 at 60 degrees",
                   abs(L.gap_fraction(0.0)) < 1e-12 and abs(L.gap_fraction(60.0) - 0.5) < 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
